# 07 — JVM, Reflection, Build & Testing

**What you'll learn:**

- The JVM's runtime memory layout: **heap**, **stack**, **metaspace**
- Garbage collection in one breath — generational hypothesis, young/old, GC choices
- How `.class` files become live classes — the **class loader** chain
- The Java Platform Module System in 60 seconds
- **Annotations revisited** — `@Retention`, `@Target`, writing your own
- **Reflection** — inspecting and invoking classes, methods, and fields at runtime
- **Reading annotations via reflection** — the mechanism Spring runs on
- **Maven** — `pom.xml`, lifecycle, dependencies, common commands
- **Gradle** — `build.gradle(.kts)`, common tasks, when each tool wins
- **JUnit 5** — `@Test`, lifecycle, assertions, parameterised tests
- **Mockito** — `mock`, `when`, `verify` for behaviour-based tests

This notebook is the bridge to Spring. Almost every Spring feature you'll meet — `@Component`, `@Autowired`, `@Transactional` — works by reading annotations off your classes at runtime via reflection and then doing something. Once you understand the JVM mechanics, Spring stops being magic and starts being inevitable.

## The JVM — what's actually running

When you launch `java MyApp`, the JVM starts up and creates a fixed set of runtime memory regions. The three you should know:

```text
   +-------------------------------------------+
   |              JVM Process                  |
   |                                           |
   |   +---------+   +---------+   +-------+  |
   |   |  Heap   |   | Stacks  |   |Meta-  |  |
   |   |         |   | (per    |   |space  |  |
   |   | objects |   |  thread)|   | class |  |
   |   |         |   | locals  |   | meta- |  |
   |   |         |   | & call  |   | data  |  |
   |   +---------+   +---------+   +-------+  |
   |                                           |
   +-------------------------------------------+
```

- **Heap** — where every object lives. Shared across all threads. Managed by the garbage collector.
- **Stack** — one per thread. Holds local variables, method parameters, and the call chain (each method call pushes a *frame*).
- **Metaspace** — class metadata: bytecode, method tables, the `Class<?>` objects themselves. Replaced the old PermGen in Java 8.

Primitives declared as local variables live on the stack. Objects always live on the heap; the local variable on the stack just holds a reference.

## Garbage collection in one breath

The JVM tracks which heap objects are reachable from your live code (local variables, static fields, etc.) and reclaims the rest. You never call `free`.

The **generational hypothesis** is the assumption every modern GC builds on: most objects die young. The heap is split into a **young generation** and an **old generation**. New objects are born in young; collections there are cheap because most objects are already dead. Objects that survive several young collections get *promoted* to old.

You don't tune GC unless something forces you to. The defaults are good. When you do tune, the three modern collectors to know:

| Collector | Pause profile | Use case |
|---|---|---|
| **G1** | Mostly short pauses | Default. General-purpose, balanced. |
| **ZGC** | Sub-millisecond pauses | Large heaps, latency-sensitive services |
| **Shenandoah** | Sub-millisecond pauses | Like ZGC; favoured in OpenJDK distributions |

Pick the collector with a JVM flag: `-XX:+UseG1GC`, `-XX:+UseZGC`, `-XX:+UseShenandoahGC`. The default since Java 9 is G1.

## Class loading

A `.class` file on disk isn't useful to the JVM until a **class loader** reads it and makes it available as a `Class<?>` object. Three loaders work in delegation:

```text
   Bootstrap loader        <- core JDK classes (java.lang.*, ...)
        |
   Platform loader         <- other JDK modules
        |
   Application loader      <- your classpath
        |
   (custom loaders)        <- frameworks, app servers, plugins
```

Class loading is **lazy** — a class is loaded the first time it's referenced. This is why Spring's "classpath scanning" works: it walks every class file it can find, asks the loader to load each one, and then uses reflection to look at the result.

You almost never write a custom class loader. But the loader chain explains a class of bugs: when two loaders load the *same* class, the JVM treats them as different types — so a cast between them fails with `ClassCastException` even though the names match.

## Java modules — the 60-second tour

Java 9 introduced the **module system** (JPMS). A module is a named, versioned bundle of packages that explicitly declares which packages it exports and which other modules it requires. A module is described by a file at the root called `module-info.java`.

In [ ]:
// module-info.java
module com.example.app {
    requires java.net.http;          // depend on a JDK module
    requires spring.context;         // depend on a library module

    exports com.example.app.api;     // make this package visible to consumers
    // packages not listed here are module-private — invisible from outside
}

Most application code, especially Spring Boot services, does **not** use JPMS directly. Spring Boot still ships as classpath JARs. JPMS matters when:

- You're building a library that wants strong encapsulation
- You're using `jlink` to ship a slimmed-down JRE with only the modules you use
- You're maintaining the JDK itself

For day-to-day Spring work, treat JPMS as background knowledge.

## Annotations — revisited and extended

Notebook 02 introduced the built-in annotations `@Override`, `@Deprecated`, `@SuppressWarnings`. Here we go deeper: **how to write your own annotation type**, and the meta-annotations that control where and when it's visible.

In [ ]:
import java.lang.annotation.*;

// where the annotation can appear
@Target({ElementType.METHOD, ElementType.TYPE})
// how long the annotation is kept
@Retention(RetentionPolicy.RUNTIME)
public @interface Audited {
    String value() default "";
    String[] tags() default {};
}

// usage
@Audited(value = "payment", tags = {"sensitive", "pii"})
public class PaymentService {

    @Audited("transfer")
    public void transfer(long fromId, long toId, long amount) {
        // ...
    }
}

The two meta-annotations that matter most:

- **`@Target`** — which element kinds the annotation can be applied to (`TYPE`, `METHOD`, `FIELD`, `PARAMETER`, `CONSTRUCTOR`, etc.)
- **`@Retention`** — how long the annotation is kept:
  - `SOURCE` — discarded at compile time. Used by lint tools.
  - `CLASS` — written to the `.class` file but not loaded at runtime. The default.
  - `RUNTIME` — present at runtime, readable via reflection. **This is what Spring needs.**

If you forget `@Retention(RetentionPolicy.RUNTIME)` on a custom annotation, reflection will not see it. This is a very common bug when people first write framework-style annotations.

## Reflection — looking at code as data

The reflection API lets your code, at runtime, inspect classes, methods, fields, constructors, and annotations — and invoke them. Everything starts from a `Class<?>` object, which you get one of three ways:

In [ ]:
// 1. from an instance
String s = "hello";
Class<?> c1 = s.getClass();

// 2. from a class literal
Class<?> c2 = String.class;

// 3. from a name (throws ClassNotFoundException)
Class<?> c3 = Class.forName("java.lang.String");

c1.getName();              // "java.lang.String"
c1.getSimpleName();        // "String"
c1.getPackageName();       // "java.lang"
c1.isInterface();          // false
c1.getSuperclass();        // class java.lang.Object
c1.getInterfaces();        // [Serializable, Comparable<String>, CharSequence]

From a `Class<?>` you can list methods, fields, and constructors, and you can invoke them.

In [ ]:
import java.lang.reflect.*;

class Greeter {
    private String prefix = "hello";
    public String greet(String name) { return prefix + ", " + name; }
}

Class<?> klass = Greeter.class;

// declared methods (this class only) vs all methods (incl. inherited public)
Method[] declared = klass.getDeclaredMethods();
Method greetMethod = klass.getDeclaredMethod("greet", String.class);

// invoking
Greeter g = (Greeter) klass.getDeclaredConstructor().newInstance();
Object result = greetMethod.invoke(g, "ganesh");    // "hello, ganesh"

// reading and writing a private field
Field prefixField = klass.getDeclaredField("prefix");
prefixField.setAccessible(true);                    // bypass private
String old = (String) prefixField.get(g);           // "hello"
prefixField.set(g, "hi");
greetMethod.invoke(g, "ganesh");                    // "hi, ganesh"

Reflection has costs — it's slower than direct calls and bypasses compile-time checks. Reach for it only when you need to operate on code you don't know about until runtime: a framework processing user-supplied classes, a serialiser inspecting fields, a test runner discovering test methods.

## Reading annotations via reflection — the Spring bridge

Here is the punchline of this whole notebook. Spring's entire programming model rests on this pattern: scan classes, find annotations on them, do something. The mechanism is two API calls.

In [ ]:
@Target({ElementType.METHOD, ElementType.TYPE})
@Retention(RetentionPolicy.RUNTIME)
@interface Audited {
    String value() default "";
}

@Audited("payment")
class PaymentService {
    @Audited("transfer")
    public void transfer(long fromId, long toId, long amount) {}
}

// inspect class-level annotation
Class<?> klass = PaymentService.class;
Audited classAnno = klass.getAnnotation(Audited.class);
classAnno.value();                                  // "payment"

// inspect method-level annotations on every method
for (Method m : klass.getDeclaredMethods()) {
    Audited a = m.getAnnotation(Audited.class);
    if (a != null) {
        System.out.println(m.getName() + " -> " + a.value());
    }
}
// transfer -> transfer

That's the seed of Spring. Spring's component scanner does exactly this, but for `@Component`, `@Service`, `@Repository`, `@Controller`, `@Configuration`. When it finds one, it instantiates the class, wires up its `@Autowired` fields by looking at their types via reflection, and registers the result in its `ApplicationContext`. Spring is not magic — it's `Class.forName`, `getAnnotation`, and `newInstance`, organised at scale.

## Maven — the dominant build tool

Almost every Spring project you encounter uses **Maven** or **Gradle**. Maven is older and more declarative — you describe what your project is, not how to build it. The project descriptor is `pom.xml`.

A minimal `pom.xml`:

In [ ]:
<project xmlns="http://maven.apache.org/POM/4.0.0">
    <modelVersion>4.0.0</modelVersion>

    <groupId>com.example</groupId>
    <artifactId>myapp</artifactId>
    <version>1.0.0</version>
    <packaging>jar</packaging>

    <properties>
        <maven.compiler.release>21</maven.compiler.release>
        <project.build.sourceEncoding>UTF-8</project.build.sourceEncoding>
    </properties>

    <dependencies>
        <dependency>
            <groupId>org.springframework.boot</groupId>
            <artifactId>spring-boot-starter-web</artifactId>
            <version>3.3.0</version>
        </dependency>
        <dependency>
            <groupId>org.junit.jupiter</groupId>
            <artifactId>junit-jupiter</artifactId>
            <version>5.10.2</version>
            <scope>test</scope>
        </dependency>
    </dependencies>
</project>

Maven runs a fixed **lifecycle**. A few commands cover daily work:

| Command | What it does |
|---|---|
| `mvn compile` | compile sources |
| `mvn test` | compile + run unit tests |
| `mvn package` | compile + test + build the JAR |
| `mvn install` | package + put the artifact in your local `~/.m2/repository` |
| `mvn clean` | delete `target/` |
| `mvn dependency:tree` | show the transitive dependency graph |

Dependencies are resolved transitively from Maven Central by default — you declare the libraries you depend on directly, and Maven pulls in everything they need.

## Gradle — the script-based alternative

Gradle uses Groovy or Kotlin DSL instead of XML, runs faster on incremental builds, and gives you full programmatic control over the build. It's the build tool Spring Boot's own initialisr defaults to today.

A minimal `build.gradle.kts` (Kotlin DSL):

In [ ]:
plugins {
    java
    id("org.springframework.boot") version "3.3.0"
    id("io.spring.dependency-management") version "1.1.5"
}

group = "com.example"
version = "1.0.0"

java {
    toolchain {
        languageVersion = JavaLanguageVersion.of(21)
    }
}

repositories {
    mavenCentral()
}

dependencies {
    implementation("org.springframework.boot:spring-boot-starter-web")
    testImplementation("org.springframework.boot:spring-boot-starter-test")
}

tasks.test {
    useJUnitPlatform()
}

Common Gradle commands:

| Command | What it does |
|---|---|
| `./gradlew build` | compile + test + package |
| `./gradlew test` | run tests |
| `./gradlew bootRun` | run a Spring Boot app (with the Boot plugin) |
| `./gradlew clean` | delete `build/` |
| `./gradlew dependencies` | show the dependency graph |

The `./gradlew` script is the **Gradle wrapper** — a tiny shim committed alongside your project that downloads the exact Gradle version the project expects. Maven has an equivalent: `./mvnw`. Always commit the wrapper; never expect contributors to have the right version installed globally.

**Maven vs Gradle, in one line:** Maven is simpler and standard; Gradle is faster and more flexible. New Spring projects increasingly start with Gradle; older codebases are almost always Maven.

## JUnit 5 — the modern test framework

JUnit 5 is the standard test runner for Java. A test class is a plain class with methods annotated `@Test`. The runner discovers them via reflection — same mechanism we just saw.

In [ ]:
import org.junit.jupiter.api.*;
import static org.junit.jupiter.api.Assertions.*;

class CalculatorTest {

    Calculator calc;

    @BeforeEach
    void setUp() {
        calc = new Calculator();
    }

    @Test
    void addsTwoNumbers() {
        assertEquals(5, calc.add(2, 3));
    }

    @Test
    void rejectsOverflow() {
        var ex = assertThrows(ArithmeticException.class,
            () -> calc.add(Integer.MAX_VALUE, 1));
        assertEquals("overflow", ex.getMessage());
    }

    @Test
    @Disabled("until bug #42 is fixed")
    void edgeCase() { /* skipped */ }
}

The lifecycle annotations:

- **`@BeforeEach`** — run before every test method
- **`@AfterEach`** — run after every test method
- **`@BeforeAll`** — run once before any test (must be static)
- **`@AfterAll`** — run once after all tests

The assertions you'll use 90% of the time: `assertEquals`, `assertTrue`/`assertFalse`, `assertNull`/`assertNotNull`, `assertThrows`, `assertAll` (group multiple checks). Reach for `assertThat` (from AssertJ) when you want fluent assertions.

## Parameterised tests

When you want to run the same test logic across many inputs, parameterise it. JUnit 5 supports several source forms; `@ValueSource` and `@CsvSource` cover most cases.

In [ ]:
import org.junit.jupiter.params.*;
import org.junit.jupiter.params.provider.*;

class StringTests {

    @ParameterizedTest
    @ValueSource(strings = {"", " ", "\t", "\n"})
    void detectsBlank(String input) {
        assertTrue(input.isBlank());
    }

    @ParameterizedTest
    @CsvSource({
        "2,  3, 5",
        "0,  0, 0",
        "-1, 1, 0"
    })
    void addsCorrectly(int a, int b, int expected) {
        assertEquals(expected, a + b);
    }
}

## Mockito — testing in isolation

When the code under test depends on something heavy — a database, a remote service, the clock — you don't want the test to need it. **Mockito** lets you substitute a fake implementation that returns whatever you tell it to, and lets you assert which methods got called.

In [ ]:
import static org.mockito.Mockito.*;

interface UserRepository {
    User findById(long id);
    void save(User user);
}

class UserService {
    private final UserRepository repo;
    public UserService(UserRepository repo) { this.repo = repo; }

    public String displayName(long id) {
        User u = repo.findById(id);
        return u == null ? "anonymous" : u.name();
    }
}

@Test
void returnsNameWhenUserExists() {
    UserRepository repo = mock(UserRepository.class);
    when(repo.findById(1L)).thenReturn(new User(1L, "alice"));

    var service = new UserService(repo);

    assertEquals("alice", service.displayName(1L));
    verify(repo).findById(1L);          // ensure the call actually happened
    verify(repo, never()).save(any());  // ensure save was NOT called
}

@Test
void returnsAnonymousWhenMissing() {
    UserRepository repo = mock(UserRepository.class);
    when(repo.findById(anyLong())).thenReturn(null);

    assertEquals("anonymous", new UserService(repo).displayName(99L));
}

The Mockito vocabulary:

- **`mock(Class)`** — create a fake. All methods return defaults (null/0/false) until you stub them.
- **`when(...).thenReturn(...)`** — define what a stubbed call returns.
- **`when(...).thenThrow(...)`** — define an exception to throw.
- **`verify(mock).method(args)`** — assert the call was made.
- **`verify(mock, never())`/`times(n)`/`atLeastOnce()`** — call-count assertions.
- **`any()`/`anyLong()`/`eq(...)`** — argument matchers.

Mockito works by generating a subclass at runtime and recording calls — yet another use of reflection. The combination of JUnit 5 + Mockito + AssertJ is the test stack you'll see in essentially every modern Spring project.

## What's next

You now know what's happening underneath your code: how the JVM lays out memory, how class loading and reflection let frameworks read your code as data, and how annotations + reflection together form the substrate Spring will use to wire your services. You also have the practical scaffolding — Maven or Gradle for builds, JUnit 5 + Mockito for tests — that every Spring project relies on.

**The Java foundation is complete.** Notebook 08 begins the Spring journey with **Spring Core**: the `ApplicationContext`, dependency injection, `@Component` scanning, bean lifecycle, configuration profiles, and AOP. Everything you saw in this notebook — annotations, reflection, classpath scanning — will resurface as Spring's machinery for getting your code to talk to itself.